In [1]:
from dataclasses import dataclass, field
from typing import Optional
from utils import hybrid_search
import json

/home/razor/Documents/alzheimers-rag/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4589.79it/s]


## Child Only Retrieval

In [2]:
@dataclass
class RetrievalConfig:
    top_k: int = 5
    alpha: float = 0.5                     # 0 = pure BM25, 1 = pure vector
    section_type_filter: Optional[str] = None
    kwd_filter: Optional[str] = None

@dataclass
class LLMConfig:
    model: str = "claude-sonnet-4-6"
    max_tokens: int = 1000
    system_prompt: str = (
        """You are a biomedical assistant synthesizing Alzheimer's disease research.

            Your job: Answer the user's question by drawing on the provided source excerpts. 
            Do NOT simply list facts from different sources. Instead:

            1. Identify the core concept the user is asking about
            2. Build a coherent narrative that explains HOW and WHY, not just WHAT
            3. Cite sources only when introducing a new empirical claim or specific finding
            4. If sources contradict or say the same thing differently, pick the clearest phrasing and cite once
            5. If the sources don't have enough information to answer the question fully, say so explicitly

            Avoid:
            - Repeating the same concept multiple times
            - Citing every sentence
            - Passive voice when active is clearer
            - Undefined jargon without explanation"""
    )

def assemble_child_context(hits):
    blocks = []
    for i, hit in enumerate(hits, 1):
        path = " > ".join(hit["heading_path"]) if hit["heading_path"] else hit["section_type"]
        blocks.append(f"[Source {i}] {hit['doc_title']} | {path}\n{hit['text']}")
    return "\n\n---\n\n".join(blocks)

def answer_with_child_context(
    query: str,
    retrieval_cfg: RetrievalConfig = field(default_factory=RetrievalConfig),
    llm_cfg: LLMConfig = field(default_factory=LLMConfig),
):
    hits = hybrid_search(
        query,
        top_k=retrieval_cfg.top_k,
        alpha=retrieval_cfg.alpha,
        section_type_filter=retrieval_cfg.section_type_filter,
        kwd_filter=retrieval_cfg.kwd_filter,
    )

    if not hits:
        return {"answer": None, "context": None, "hits": [], "note": "No chunks retrieved"}

    context = assemble_child_context(hits)
    user_message = f"Context:\n\n{context}\n\n---\n\nQuestion: {query}"

    return {
        "answer": user_message,
        "context": context,
        "hits": hits,
        "num_chunks_used": len(hits),
        "retrieval_cfg": retrieval_cfg,   # stash the config used — you'll want this logged for eval comparisons
        "llm_cfg": llm_cfg,
    }

In [30]:
query = "How do recent PMC studies differentiate the structural characteristics of neuritic plaques from diffuse plaques in post-mortem Alzheimer's brain tissue?"

result = answer_with_child_context(
    query,
    retrieval_cfg=RetrievalConfig(top_k=5, alpha=0.5),
)

print("ANSWER:\n", result["answer"])
print("\nCONFIG USED:", result["retrieval_cfg"])

ANSWER:
 Context:

[Source 1]  | Methods > Human brain tissue selection and preparation.
Specimens were obtained from the Banner Sun Health Research Institute brain bank from the Arizona Study of Aging and Neurodegenerative Disease (AZSAND). Donors enrolled in the study underwent comprehensive clinical assessments during life, which included neurological exams, neuropsychological tests, and medical record reviews. Upon death, the autopsy protocol is initiated with a median post-mortem interval (PMI) of 3 hours to optimize tissue preservation. Brain extraction was performed using an oscillating electric saw to open the calvarium and expose the intact brain, followed by the extraction of the brain with the olfactory bulbs and pituitary gland still intact. Gross pathological features are then documented and photographed. The brain is then immediately divided so that the cerebellum and brainstem are separated from the cerebrum, and the cerebrum is coronally sectioned into 1cm slices using 

In [33]:
no_rerank = [(item["chunk_id"], item["text"]) for item in result["hits"]]

## Parent-Child Retrieval

In [4]:
# --- Load parents once into memory ---
PARENTS_JSONL = "/home/razor/Documents/alzheimers-rag/data/processed/pmc_alzheimers_parents.jsonl"

In [5]:
def load_parent_lookup(path):
    lookup = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            record = json.loads(line)
            lookup[record["parent_id"]] = record
    return lookup

PARENT_LOOKUP = load_parent_lookup(PARENTS_JSONL)
print(f"Loaded {len(PARENT_LOOKUP)} parents into memory")

@dataclass
class RetrievalConfig:
    top_k: int = 5
    alpha: float = 0.5
    section_type_filter: Optional[str] = None
    kwd_filter: Optional[str] = None

@dataclass
class LLMConfig:
    model: str = "claude-sonnet-4-6"
    max_tokens: int = 1000
    system_prompt: str = (
        """You are a biomedical assistant synthesizing Alzheimer's disease research.

            Your job: Answer the user's question by drawing on the provided source excerpts. 
            Do NOT simply list facts from different sources. Instead:

            1. Identify the core concept the user is asking about
            2. Build a coherent narrative that explains HOW and WHY, not just WHAT
            3. Cite sources only when introducing a new empirical claim or specific finding
            4. If sources contradict or say the same thing differently, pick the clearest phrasing and cite once
            5. If the sources don't have enough information to answer the question fully, say so explicitly

            Avoid:
            - Repeating the same concept multiple times
            - Citing every sentence
            - Passive voice when active is clearer
            - Undefined jargon without explanation"""
    )

def assemble_parent_context(hits):
    """
    Dedupes child hits by parent_id — fetches each distinct parent once,
    ordered by that parent's best-scoring child hit.
    If a child_id has no matching parent (shouldn't happen, but don't fail silently),
    falls back to the child's own text with a warning marker.
    """
    seen_parents = {}   # parent_id -> best score seen so far
    parent_order = []

    for hit in hits:
        pid = hit.get("chunk_id") and hit.get("parent_id")  # just being explicit this comes from the child record
        pid = hit["parent_id"] if "parent_id" in hit else None
        if pid is None:
            continue
        if pid not in seen_parents or hit["score"] > seen_parents[pid]:
            seen_parents[pid] = hit["score"]
        if pid not in parent_order:
            parent_order.append(pid)

    # sort distinct parents by their best child score, descending
    parent_order.sort(key=lambda pid: seen_parents[pid], reverse=True)

    blocks = []
    missing = []
    for i, pid in enumerate(parent_order, 1):
        parent = PARENT_LOOKUP.get(pid)
        if parent is None:
            missing.append(pid)
            continue
        path = " > ".join(parent.get("heading_path", [])) if parent.get("heading_path") else parent.get("section_type", "")
        block = (
            f"[Source {i}] {parent.get('doc_title', '')} | {path}\n"
            f"{parent['text']}"
        )
        blocks.append(block)

    if missing:
        print(f"WARNING: {len(missing)} parent_id(s) not found in lookup — check data integrity: {missing}")

    return "\n\n---\n\n".join(blocks), len(parent_order) - len(missing)

def answer_with_parent_context(
    query: str,
    retrieval_cfg: RetrievalConfig = field(default_factory=RetrievalConfig),
    llm_cfg: LLMConfig = field(default_factory=LLMConfig),
):
    hits = hybrid_search(
        query,
        top_k=retrieval_cfg.top_k,
        alpha=retrieval_cfg.alpha,
        section_type_filter=retrieval_cfg.section_type_filter,
        kwd_filter=retrieval_cfg.kwd_filter,
    )

    if not hits:
        print("No hits")
        return {"answer": None, "context": None, "hits": [], "note": "No chunks retrieved"}

    context, num_parents_used = assemble_parent_context(hits)

    if not context:
        return {"answer": None, "context": None, "hits": hits, "note": "No valid parents resolved from hits"}

    user_message = f"{llm_cfg.system_prompt} \n\nContext:\n\n{context}\n\n---\n\nQuestion: {query}"

    return {
        "answer": user_message,
        "context": context,
        "hits": hits,
        "num_children_retrieved": len(hits),
        "num_parents_used": num_parents_used,   # tells you how much dedup collapsed the set — worth watching
        "retrieval_cfg": retrieval_cfg,
        "llm_cfg": llm_cfg,
    }

Loaded 46301 parents into memory


In [12]:
query = "What is the role of amyloid-beta plaques in Alzheimer's progression?"
result = answer_with_parent_context(query, retrieval_cfg=RetrievalConfig(top_k=5), llm_cfg=LLMConfig())

if result["answer"] is not None:
    print("ANSWER:\n", result["answer"])
    print(f"\nChildren retrieved: {result['num_children_retrieved']} → Parents used: {result['num_parents_used']}")

ANSWER:
 You are a biomedical assistant synthesizing Alzheimer's disease research.

            Your job: Answer the user's question by drawing on the provided source excerpts. 
            Do NOT simply list facts from different sources. Instead:

            1. Identify the core concept the user is asking about
            2. Build a coherent narrative that explains HOW and WHY, not just WHAT
            3. Cite sources only when introducing a new empirical claim or specific finding
            4. If sources contradict or say the same thing differently, pick the clearest phrasing and cite once
            5. If the sources don't have enough information to answer the question fully, say so explicitly

            Avoid:
            - Repeating the same concept multiple times
            - Citing every sentence
            - Passive voice when active is clearer
            - Undefined jargon without explanation 

Context:

[Source 1]  | 1. Introduction
Alzheimer’s disease (AD) is chara

## Evaluation

In [ ]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1")
token_list = encoding.encode(result["answer"])
token_count = len(token_list)

print("Token count:", token_count)

In [15]:
import csv

# --- Config ---
EVAL_QUESTIONS_PATH = "eval_questions.jsonl"   # you write this: [{"question_id": "q1", "question": "..."}]
POOL_TOP_K = 10                               # generous — this is the candidate pool, not final retrieval
OUTPUT_JUDGMENT_SHEET = "judgment_sheet.csv"

def load_questions(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def build_pool_for_question(question_id, question_text, top_k=POOL_TOP_K):
    """
    Retrieves a generous top_k of child hits, then derives BOTH:
    - child-level candidates (raw hits)
    - parent-level candidates (deduped by parent_id, same as assemble_parent_context logic)
    Both get judged in the same sheet so one labeling pass covers both eval tracks.
    """
    hits = hybrid_search(question_text, top_k=top_k, alpha=0.5)  # fixed alpha for pooling — not what we're testing here

    rows = []

    # child-level rows
    for rank, hit in enumerate(hits, 1):
        rows.append({
            "question_id": question_id,
            "question": question_text,
            "granularity": "child",
            "candidate_id": hit["chunk_id"],
            "rank": rank,
            "score": round(hit["score"], 4),
            "doc_title": hit["doc_title"],
            "text_preview": hit["text"][:300].replace("\n", " "),
            "relevant": "",   # <-- you fill this in: 1 = relevant, 0 = not, blank = unsure
        })

    # parent-level rows — dedup by parent_id, ranked by best child score
    seen_parents = {}
    parent_order = []
    for hit in hits:
        pid = hit["parent_id"]
        if pid not in seen_parents or hit["score"] > seen_parents[pid]:
            seen_parents[pid] = hit["score"]
        if pid not in parent_order:
            parent_order.append(pid)
    parent_order.sort(key=lambda pid: seen_parents[pid], reverse=True)

    for rank, pid in enumerate(parent_order, 1):
        parent = PARENT_LOOKUP.get(pid)
        if parent is None:
            continue
        rows.append({
            "question_id": question_id,
            "question": question_text,
            "granularity": "parent",
            "candidate_id": pid,
            "rank": rank,
            "score": round(seen_parents[pid], 4),
            "doc_title": parent.get("doc_title", ""),
            "text_preview": parent["text"][:300].replace("\n", " "),
            "relevant": "",
        })

    return rows

def main():
    questions = load_questions(EVAL_QUESTIONS_PATH)
    print(f"Building pools for {len(questions)} questions, top_k={POOL_TOP_K}")

    all_rows = []
    for q in questions:
        rows = build_pool_for_question(q["question_id"], q["question"])
        all_rows.extend(rows)
        print(f"  {q['question_id']}: {len(rows)} candidates pooled")

    fieldnames = ["question_id", "question", "granularity", "candidate_id", "rank", "score", "doc_title", "text_preview", "relevant"]
    with open(OUTPUT_JUDGMENT_SHEET, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\nWrote {len(all_rows)} rows to {OUTPUT_JUDGMENT_SHEET}")
    print("Open in Excel/Sheets, sort by question_id, fill 'relevant' column with 1/0.")

if __name__ == "__main__":
    main()

Building pools for 10 questions, top_k=10
  q1: 20 candidates pooled
  q2: 20 candidates pooled
  q3: 20 candidates pooled
  q4: 20 candidates pooled
  q5: 20 candidates pooled
  q6: 18 candidates pooled
  q7: 19 candidates pooled
  q8: 20 candidates pooled
  q9: 18 candidates pooled
  q10: 20 candidates pooled

Wrote 195 rows to judgment_sheet.csv
Open in Excel/Sheets, sort by question_id, fill 'relevant' column with 1/0.


In [8]:
import csv
from collections import defaultdict

JUDGMENT_SHEET = "judgment_sheet.csv"
K_VALUES = [3, 5, 10]   # compute recall at multiple cutoffs since you haven't picked a final top_k yet

def load_judgments(path):
    """
    Groups rows by (question_id, granularity), sorted by rank.
    Skips rows where 'relevant' is blank (unjudged) — flags them rather than silently treating as 0.
    """
    grouped = defaultdict(list)
    skipped = 0

    with open(path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            if row["relevant"].strip() == "":
                skipped += 1
                continue
            key = (row["question_id"], row["granularity"])
            grouped[key].append({
                "candidate_id": row["candidate_id"],
                "rank": int(row["rank"]),
                "relevant": int(row["relevant"]),
            })

    if skipped:
        print(f"WARNING: {skipped} unjudged rows skipped (blank 'relevant' field) — these are NOT counted as irrelevant, just excluded")

    for key in grouped:
        grouped[key].sort(key=lambda r: r["rank"])

    return grouped

def recall_at_k(candidates, k):
    total_relevant = sum(1 for c in candidates if c["relevant"] == 1)
    if total_relevant == 0:
        return None  # no relevant docs judged for this question — can't compute, don't silently return 0
    retrieved_at_k = [c for c in candidates if c["rank"] <= k]
    hits = sum(1 for c in retrieved_at_k if c["relevant"] == 1)
    return hits / total_relevant

def mrr(candidates):
    for c in candidates:
        if c["relevant"] == 1:
            return 1.0 / c["rank"]
    return 0.0  # no relevant doc found anywhere in the pool

def compute_metrics(grouped):
    results = defaultdict(lambda: defaultdict(list))  # granularity -> metric -> list of per-question values

    for (question_id, granularity), candidates in grouped.items():
        for k in K_VALUES:
            r = recall_at_k(candidates, k)
            if r is not None:
                results[granularity][f"recall@{k}"].append(r)
            else:
                print(f"  Skipping recall@{k} for {question_id}/{granularity}: no relevant docs judged")

        results[granularity]["mrr"].append(mrr(candidates))

    return results

def print_summary(results):
    for granularity in results:
        print(f"\n=== {granularity.upper()} ===")
        for metric, values in results[granularity].items():
            avg = sum(values) / len(values)
            print(f"  {metric}: {avg:.3f}  (n={len(values)})")

if __name__ == "__main__":
    grouped = load_judgments(JUDGMENT_SHEET)
    print(f"Loaded judgments for {len(set(k[0] for k in grouped))} questions")

    results = compute_metrics(grouped)
    print_summary(results)

Loaded judgments for 7 questions
  Skipping recall@3 for q3/child: no relevant docs judged
  Skipping recall@5 for q3/child: no relevant docs judged
  Skipping recall@10 for q3/child: no relevant docs judged
  Skipping recall@3 for q3/parent: no relevant docs judged
  Skipping recall@5 for q3/parent: no relevant docs judged
  Skipping recall@10 for q3/parent: no relevant docs judged

=== CHILD ===
  recall@3: 0.571  (n=6)
  recall@5: 0.692  (n=6)
  recall@10: 1.000  (n=6)
  mrr: 0.714  (n=7)

=== PARENT ===
  recall@3: 0.551  (n=6)
  recall@5: 0.672  (n=6)
  recall@10: 1.000  (n=6)
  mrr: 0.714  (n=7)


## Re-Ranking

In [6]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from dataclasses import dataclass

RERANK_MODEL_NAME = "ncbi/MedCPT-Cross-Encoder"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

rerank_tokenizer = AutoTokenizer.from_pretrained(RERANK_MODEL_NAME)
rerank_model = AutoModelForSequenceClassification.from_pretrained(RERANK_MODEL_NAME).to(DEVICE)
rerank_model.eval()

@dataclass
class RerankConfig:
    candidate_pool_size: int = 30   # how many hybrid hits to pull before reranking
    rerank_top_k: int = 5           # how many survive reranking, passed to context assembly
    batch_size: int = 16            # cross-encoder batch size for scoring pairs

def rerank_hits(query: str, hits: list, cfg: RerankConfig = RerankConfig()):
    """
    Takes hybrid_search() output (already up to candidate_pool_size in length —
    caller is responsible for retrieving that many) and reorders by cross-encoder score.
    Returns top rerank_top_k hits, each with an added 'rerank_score' field.
    """
    if not hits:
        return hits

    pairs = [[query, hit["text"]] for hit in hits]
    all_scores = []

    with torch.no_grad():
        for i in range(0, len(pairs), cfg.batch_size):
            batch_pairs = pairs[i:i + cfg.batch_size]
            encoded = rerank_tokenizer(
                batch_pairs,
                truncation=True,
                padding=True,
                max_length=512,
                return_tensors="pt",
            ).to(DEVICE)
            logits = rerank_model(**encoded).logits.squeeze(-1)  # shape: (batch,)
            all_scores.extend(logits.cpu().tolist())

    for hit, score in zip(hits, all_scores):
        hit["rerank_score"] = score
        hit["original_hybrid_score"] = hit["score"]   # preserve original — useful diagnostic, don't overwrite silently

    reranked = sorted(hits, key=lambda h: h["rerank_score"], reverse=True)
    return reranked[:cfg.rerank_top_k]


def hybrid_search_with_rerank(query, retrieval_cfg: RetrievalConfig, rerank_cfg: RerankConfig):
    """
    Drop-in replacement flow: retrieve a wide pool via hybrid search,
    rerank down to the final set. Feed this into either
    assemble_child_context() or assemble_parent_context() same as before.
    """
    wide_hits = hybrid_search(
        query,
        top_k=rerank_cfg.candidate_pool_size,   # note: pool size, not final top_k
        alpha=retrieval_cfg.alpha,
        section_type_filter=retrieval_cfg.section_type_filter,
        kwd_filter=retrieval_cfg.kwd_filter,
    )

    if not wide_hits:
        return []

    reranked_hits = rerank_hits(query, wide_hits, rerank_cfg)
    return reranked_hits

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 13596.57it/s]


In [29]:
query = "How do recent PMC studies differentiate the structural characteristics of neuritic plaques from diffuse plaques in post-mortem Alzheimer's brain tissue?"

retrieval_cfg = RetrievalConfig(alpha=0.5)
rerank_cfg = RerankConfig(candidate_pool_size=30, rerank_top_k=5)

reranked_hits = hybrid_search_with_rerank(query, retrieval_cfg, rerank_cfg)

for i, hit in enumerate(reranked_hits, 1):
    print(f"[{i}] rerank_score={hit['rerank_score']:.4f} | hybrid_score={hit['original_hybrid_score']:.4f}")
    print(f"    {hit['doc_title']} | {hit['text'][:150]}...\n")

[1] rerank_score=-0.3232 | hybrid_score=0.4816
     | The inclusion of all forms of p-tau which AT8 captures in our model (including pre-tangles, NFTs, p-tau-positive neurites, and neuritic plaques) also ...

[2] rerank_score=-5.2988 | hybrid_score=0.2883
     | Tau pathology is universal in aging, though the patterns of tau pathology have been reported to differ between AD and PART in older individuals. Here ...

[3] rerank_score=-5.4045 | hybrid_score=0.2875
     | Rather, examination of the standardized coefficients showed that the sleep fragmentation proximal to death may have a stronger association with amyloi...

[4] rerank_score=-5.7156 | hybrid_score=0.2577
     | Alzheimer's disease (ad) is a neurodegenerative disorder that leads to cognitive impairment, functional decline and neuropsychiatric symptoms. ad affe...

[5] rerank_score=-6.3728 | hybrid_score=0.2661
     | Fluid and imaging biomarker data show that anti-amyloid (Aβ) antibodies promote the clearance of Aβ from the b

In [31]:
with_rerank = [(item["chunk_id"], item["text"]) for item in reranked_hits]

In [35]:
no_rerank, with_rerank

([('PMC13228798_s3_c0',
   'Specimens were obtained from the Banner Sun Health Research Institute brain bank from the Arizona Study of Aging and Neurodegenerative Disease (AZSAND). Donors enrolled in the study underwent comprehensive clinical assessments during life, which included neurological exams, neuropsychological tests, and medical record reviews. Upon death, the autopsy protocol is initiated with a median post-mortem interval (PMI) of 3 hours to optimize tissue preservation. Brain extraction was performed using an oscillating electric saw to open the calvarium and expose the intact brain, followed by the extraction of the brain with the olfactory bulbs and pituitary gland still intact. Gross pathological features are then documented and photographed. The brain is then immediately divided so that the cerebellum and brainstem are separated from the cerebrum, and the cerebrum is coronally sectioned into 1cm slices using a custom slicing jig. The hippocampus is systemically sampled